# Bolt Plus A/B test scorecard — dynamic

Fill in the widgets at the top, then **Run All**. Nothing below needs editing.

| Widget | Required | What happens if you leave it blank |
|---|---|---|
| `test_ids` | **yes** | — one or more test IDs, comma separated |
| `test_start` | no | taken from the enrollment table |
| `test_end` | no | taken from the enrollment table, capped at today |
| `enrolled_from` | no | auto: 60 days before `test_start`, else `2025-01-01` |
| `treatment_group_ids` | no | arm inferred from the enrollment table |
| `control_group_ids` | no | arm inferred from the enrollment table |

## How the test ID is resolved

The test IDs are looked up in **both** enrollment ETLs and whichever one holds the test is used, so
it does not matter which platform the test ran on.

| Source | Table | Holds | User key | Arm column |
|---|---|---|---|---|
| `delivery` | `main.ng_public.etl_delivery_campaigns_enrollments` | eaters / delivery | `eater_id` | `treatment_type` — a real `treatment`/`control` flag |
| `incentives` | `main.core_models.dim_incentives_enrollment` | rides, supply, cars, business | `campaign_enrollment_target_id` | `campaign_treatment_type` — a **product** type, not an arm |

The two tables use separate test-ID namespaces, so an ID resolves to one of them and the other
branch returns nothing.

Because `campaign_treatment_type` is a product type (`subscription_campaign`, `targeted_campaign`,
`targeted_control_campaign`, `comms_only_campaign`, …), the notebook treats anything matching
`control` as control and any other non-null type as treatment. Use the group-ID widgets to override
that if your test does not fit the pattern.

Cell 4 is the resolution map and cell 5 is a validation query that must return **zero rows** before
the numbers below mean anything.

In [ ]:
# ===========================================================================
# 1. Widgets. Set these, then Run All.
# ===========================================================================
DEFAULTS = {
    "test_ids": "6625258,6413890",
    "test_start": "",
    "test_end": "",
    "enrolled_from": "",
    "treatment_group_ids": "",
    "control_group_ids": "",
}

try:
    dbutils.widgets.text("test_ids", DEFAULTS["test_ids"], "1. Test IDs (comma separated)")
    dbutils.widgets.text("test_start", DEFAULTS["test_start"], "2. Test start YYYY-MM-DD (blank = from table)")
    dbutils.widgets.text("test_end", DEFAULTS["test_end"], "3. Test end YYYY-MM-DD (blank = from table)")
    dbutils.widgets.text("enrolled_from", DEFAULTS["enrolled_from"], "4. Enrollment scan floor (blank = auto)")
    dbutils.widgets.text("treatment_group_ids", DEFAULTS["treatment_group_ids"], "5. Force treatment group IDs")
    dbutils.widgets.text("control_group_ids", DEFAULTS["control_group_ids"], "6. Force control group IDs")

    def _raw(name):
        return (dbutils.widgets.get(name) or "").strip()
except NameError:                                    # not running on Databricks
    def _raw(name):
        return DEFAULTS[name].strip()


import datetime as _dt
import re as _re


def _ints(raw):
    return [int(p) for p in raw.replace(";", ",").split(",") if p.strip()]


def _date(raw, label):
    if not raw:
        return None
    if not _re.fullmatch(r"\d{4}-\d{2}-\d{2}", raw):
        raise ValueError(f"{label} must be YYYY-MM-DD, got {raw!r}")
    return _dt.date.fromisoformat(raw)


TEST_IDS = _ints(_raw("test_ids"))
TEST_START = _date(_raw("test_start"), "test_start")
TEST_END = _date(_raw("test_end"), "test_end")
FORCE_TREATMENT = _ints(_raw("treatment_group_ids"))
FORCE_CONTROL = _ints(_raw("control_group_ids"))

if not TEST_IDS:
    raise ValueError("test_ids is required")
if TEST_START and TEST_END and TEST_END < TEST_START:
    raise ValueError(f"test_end {TEST_END} is before test_start {TEST_START}")
if set(FORCE_TREATMENT) & set(FORCE_CONTROL):
    raise ValueError(f"group(s) {sorted(set(FORCE_TREATMENT) & set(FORCE_CONTROL))} are in both arms")

# The enrollment scan floor only prunes partitions; it never widens the analysis
# window. Enrollments land close to test start, so 60 days back is generous.
ENROLLED_FROM = _date(_raw("enrolled_from"), "enrolled_from")
if ENROLLED_FROM is None:
    ENROLLED_FROM = TEST_START - _dt.timedelta(days=60) if TEST_START else _dt.date(2025, 1, 1)

print(f"test ids         : {TEST_IDS}")
print(f"test start       : {TEST_START or 'from enrollment table'}")
print(f"test end         : {TEST_END or 'from enrollment table (capped at today)'}")
print(f"enrollment floor : {ENROLLED_FROM}")
print(f"forced treatment : {FORCE_TREATMENT or '—'}")
print(f"forced control   : {FORCE_CONTROL or '—'}")

In [ ]:
# ===========================================================================
# 2. Build every view from the widgets. This is the whole engine; the %sql
#    cells below just read the views it creates.
# ===========================================================================
DELIVERY_ENROLL = "main.ng_public.etl_delivery_campaigns_enrollments"
INCENTIVE_ENROLL = "main.core_models.dim_incentives_enrollment"
MONETARY = "main.ng_public.etl_delivery_order_monetary_metrics"
SUBSCRIPTIONS = "main.core_models.fact_user_subscriptions"
LIFECYCLE = "main.ng_delivery.int_user_metrics_user_cohort"

COUNTRY_NAMES = {
    "ee": "Estonia", "lv": "Latvia", "lt": "Lithuania", "pl": "Poland", "ro": "Romania",
    "bg": "Bulgaria", "hr": "Croatia", "cz": "Czechia", "sk": "Slovakia", "hu": "Hungary",
    "pt": "Portugal", "es": "Spain", "se": "Sweden", "fi": "Finland", "de": "Germany",
    "at": "Austria", "nl": "Netherlands", "fr": "France", "gb": "United Kingdom",
    "ua": "Ukraine", "rs": "Serbia", "ge": "Georgia", "cy": "Cyprus", "mt": "Malta",
    "ng": "Nigeria", "za": "South Africa", "ke": "Kenya", "gh": "Ghana", "ug": "Uganda",
    "tz": "Tanzania", "tn": "Tunisia", "ma": "Morocco", "iq": "Iraq", "tr": "Turkey",
    "az": "Azerbaijan", "mx": "Mexico", "no": "Norway", "dk": "Denmark", "ie": "Ireland",
}

IDS = ", ".join(str(i) for i in TEST_IDS)


def _in(values):
    return "(" + ", ".join(str(v) for v in values) + ")"


# A typed test-start / test-end override replaces the table window verbatim.
START_EXPR = f"timestamp'{TEST_START} 00:00:00'" if TEST_START else "min(e.test_start)"
END_EXPR = f"timestamp'{TEST_END} 23:59:59'" if TEST_END else "max(e.test_end)"

_override = ""
if FORCE_TREATMENT:
    _override += f"\n      when group_id in {_in(FORCE_TREATMENT)} then 'treatment'"
if FORCE_CONTROL:
    _override += f"\n      when group_id in {_in(FORCE_CONTROL)} then 'control'"

spark.sql(
    "create or replace temporary view markets as select * from values "
    + ", ".join(f"('{cc}', '{name}')" for cc, name in COUNTRY_NAMES.items())
    + " as t(country_code, market)"
)

# Requested IDs kept as a view so a test found in neither table still shows up.
spark.sql(
    "create or replace temporary view requested_tests as select * from values "
    + ", ".join(f"(cast({i} as bigint))" for i in TEST_IDS)
    + " as t(test_id)"
)

spark.sql(f"""
create or replace temporary view enrollment as
select
  *,
  case{_override}
    when lower(trim(arm_label)) rlike 'control'          then 'control'
    when arm_label is not null and trim(arm_label) <> '' then 'treatment'
    when source = 'delivery' and campaign_id is null     then 'control'
  end as variant
from (
  -- Eaters / delivery platform: treatment_type is a real arm flag.
  select
    'delivery'                     as source,
    cast(eater_id as string)       as user_id,
    cast(test_id as bigint)        as test_id,
    cast(treatment_id as bigint)   as group_id,
    cast(campaign_id as bigint)    as campaign_id,
    cast(treatment_type as string) as arm_label,
    lower(trim(country))           as country_code,
    cast(test_start as timestamp)  as test_start,
    cast(test_end as timestamp)    as test_end,
    cast(enrolled_at as timestamp) as enrolled_ts
  from {DELIVERY_ENROLL}
  where test_id in ({IDS})
    and enrolled_at_date >= date'{ENROLLED_FROM}'

  union all

  -- Rides / supply / cars / business: campaign_treatment_type is a product type,
  -- so 'control' in the name is the only reliable control marker.
  select
    'incentives',
    cast(campaign_enrollment_target_id as string),
    cast(campaign_test_id as bigint),
    cast(campaign_treatment_id as bigint),
    cast(null as bigint),
    cast(campaign_treatment_type as string),
    lower(trim(campaign_test_country_code)),
    cast(campaign_test_start_ts as timestamp),
    cast(campaign_test_end_ts as timestamp),
    cast(campaign_enrollment_created_ts as timestamp)
  from {INCENTIVE_ENROLL}
  where campaign_test_id in ({IDS})
    and campaign_enrollment_created_date >= date'{ENROLLED_FROM}'
)
where user_id is not null
""")

if spark.sql("select 1 from enrollment limit 1").count() == 0:
    raise ValueError(
        f"no enrollments found for {TEST_IDS} in either table. Check the IDs, or set "
        f"enrolled_from earlier than {ENROLLED_FROM}."
    )

# One row per user per arm per country: a user can be in several of the selected
# tests, and their orders and subscriptions must only be counted once.
spark.sql(f"""
create or replace temporary view cohort as
select
  e.user_id,
  e.variant,
  e.country_code,
  coalesce(m.market, upper(e.country_code))          as market,
  {START_EXPR}                                       as test_start,
  least({END_EXPR}, current_timestamp())             as window_end
from enrollment e
left join markets m on m.country_code = e.country_code
where e.variant is not null
group by e.user_id, e.variant, e.country_code, coalesce(m.market, upper(e.country_code))
""")

_b = spark.sql("""
select min(to_date(test_start)) as win_start,
       max(to_date(window_end)) as win_end,
       max(to_date(test_start)) as last_start,
       count(*)                 as users
from cohort
""").first()
WIN_START, WIN_END, LAST_START = _b["win_start"], _b["win_end"], _b["last_start"]

if WIN_START is None or WIN_END is None:
    raise ValueError("cohort has no usable test window — check cell 5 (validation)")
if WIN_END < WIN_START:
    raise ValueError(f"test window is empty: {WIN_START} → {WIN_END}")

COUNTRIES = [r["country_code"] for r in spark.sql("select distinct country_code from cohort").collect()]
CC_SQL = ", ".join(f"'{c}'" for c in COUNTRIES)

# Window and countries as a view, so the %sql cells can reference them.
spark.sql(f"""
create or replace temporary view bounds as
select date'{WIN_START}' as win_start, date'{WIN_END}' as win_end
""")

# Delivery orders inside each user's own window. The outer date predicate is a
# literal so the partitioned table prunes; the inner one is the per-user window.
spark.sql(f"""
create or replace temporary view cohort_orders as
select
  c.user_id, c.market, c.variant, c.country_code,
  count(distinct o.order_id)                 as orders,
  sum(o.gmv_eur)                             as gmv,
  sum(o.nmv_eur)                             as nmv,
  sum(o.net_income_eur)                      as net_income,
  sum(o.total_discount_eur)                  as discount,
  sum(o.provider_commission_gross_eur)       as provider_commission,
  sum(o.bolt_delivery_campaign_cost_eur)     as bolt_delivery_cost,
  sum(o.bolt_menu_campaign_cost_eur)         as bolt_menu_cost,
  sum(o.provider_delivery_campaign_cost_eur) as provider_delivery_cost,
  sum(o.provider_menu_campaign_cost_eur)     as provider_menu_cost,
  sum(o.delivery_discount_eur)               as delivery_discount,
  sum(o.menu_discount_eur)                   as menu_discount
from cohort c
join {MONETARY} o
  on cast(o.user_id as string) = c.user_id
 and lower(o.country) = c.country_code
 and o.order_created_date between to_date(c.test_start) and to_date(c.window_end)
where o.order_created_date between date'{WIN_START}' and date'{WIN_END}'
  and lower(o.country) in ({CC_SQL})
group by c.user_id, c.market, c.variant, c.country_code
""")

# Order dates, for weekly behavioural retention.
spark.sql(f"""
create or replace temporary view cohort_order_days as
select distinct
  c.user_id, c.market, c.variant,
  to_date(c.test_start) as test_start_date,
  o.order_created_date
from cohort c
join {MONETARY} o
  on cast(o.user_id as string) = c.user_id
 and lower(o.country) = c.country_code
 and o.order_created_date between to_date(c.test_start) and to_date(c.window_end)
where o.order_created_date between date'{WIN_START}' and date'{WIN_END}'
  and lower(o.country) in ({CC_SQL})
""")

# Order frequency in the 90 days before each user's test start.
spark.sql(f"""
create or replace temporary view cohort_prior_orders as
select c.user_id, c.market, c.variant, count(distinct o.order_id) as prior_orders
from cohort c
left join {MONETARY} o
  on cast(o.user_id as string) = c.user_id
 and lower(o.country) = c.country_code
 and o.order_created_date between date_sub(to_date(c.test_start), 90)
                              and date_sub(to_date(c.test_start), 1)
 and o.order_created_date between date_sub(date'{WIN_START}', 90) and date'{LAST_START}'
 and lower(o.country) in ({CC_SQL})
group by c.user_id, c.market, c.variant
""")

# Subscriptions started inside each user's own window.
spark.sql(f"""
create or replace temporary view cohort_subs as
select
  c.user_id, c.market, c.variant, c.country_code, c.test_start,
  s.subscription_start_time,
  s.subscription_end_time,
  s.subscription_started_with_trial,
  s.subscription_period_type,
  s.subscription_price_discounted_local
from cohort c
join {SUBSCRIPTIONS} s
  on cast(s.user_id as string) = c.user_id
 and s.subscription_start_time between c.test_start and c.window_end
where s.subscription_start_date_utc between date'{WIN_START}' and date'{WIN_END}'
""")

# Lifecycle snapshot the day before test start. This table is delivery-only, so a
# rides-domain test simply comes back as 'Unknown'.
spark.sql(f"""
create or replace temporary view cohort_lifecycle as
select
  cast(l.user_id as string) as user_id,
  lower(l.country_code)     as country_code,
  l.metric_date,
  l.user_cohort             as prior_stage,
  l.user_sub_cohort         as prior_cohort_name
from {LIFECYCLE} l
where l.metric_date in (select distinct date_sub(to_date(test_start), 1) from cohort)
  and lower(l.country_code) in ({CC_SQL})
""")

print(f"window    : {WIN_START} → {WIN_END}  ({(WIN_END - WIN_START).days} days)")
print(f"markets   : {[COUNTRY_NAMES.get(c, c.upper()) for c in COUNTRIES]}")
print(f"cohort    : {_b['users']:,} users")
for _v in ("cohort_orders", "cohort_order_days", "cohort_prior_orders", "cohort_subs", "cohort_lifecycle"):
    print(f"{_v:22s}{spark.sql(f'select count(*) n from {_v}').first()['n']:>12,} rows")

In [ ]:
%sql
-- ===========================================================================
-- 3. Test resolution map — which ETL each requested test came from.
--    A test ID showing 'NOT FOUND' is in neither table.
-- ===========================================================================
select
  r.test_id,
  coalesce(e.source, 'NOT FOUND IN EITHER TABLE') as source,
  coalesce(e.market, '—')                         as market,
  coalesce(e.country_code, '—')                   as country_code,
  coalesce(e.variant, 'UNCLASSIFIED')             as variant,
  e.group_id,
  e.arm_label,
  e.users,
  date(e.test_start)                              as test_start,
  date(e.test_end)                                as test_end,
  date(e.first_enrolled_ts)                       as first_enrolled_date
from requested_tests r
left join (
  select
    n.source, n.test_id, n.group_id, n.variant, n.arm_label, n.country_code,
    coalesce(m.market, upper(n.country_code)) as market,
    count(distinct n.user_id)                 as users,
    min(n.test_start)                         as test_start,
    max(n.test_end)                           as test_end,
    min(n.enrolled_ts)                        as first_enrolled_ts
  from enrollment n
  left join markets m on m.country_code = n.country_code
  group by n.source, n.test_id, n.group_id, n.variant, n.arm_label, n.country_code,
           coalesce(m.market, upper(n.country_code))
) e on e.test_id = r.test_id
order by r.test_id, source, market, variant, e.group_id

In [ ]:
%sql
-- ===========================================================================
-- 4. Validation. An EMPTY result is the pass state.
--    Anything returned here means the numbers below are not trustworthy.
-- ===========================================================================
select 'test_id_not_found' as issue,
       cast(r.test_id as string) as detail,
       1 as rows_affected,
       'ID is in neither table — check it, or set enrolled_from earlier' as fix
from requested_tests r
left join enrollment e on e.test_id = r.test_id
where e.test_id is null

union all
select 'unclassified_arm',
       concat(source, ' test ', test_id, ' group ',
              coalesce(cast(group_id as string), 'null'),
              ' label ', coalesce(arm_label, 'null')),
       count(distinct user_id),
       'set the treatment_group_ids / control_group_ids widgets'
from enrollment
where variant is null
group by source, test_id, group_id, arm_label

union all
select 'missing_window_or_country',
       concat(source, ' test ', test_id), count(*),
       'test_start / test_end / country is null on the enrollment rows'
from enrollment
where country_code is null or test_start is null or test_end is null
group by source, test_id

union all
select 'user_in_both_arms', 'combined', count(*),
       'these users are double counted — drop one test or exclude them'
from (select user_id from enrollment where variant is not null
      group by user_id having count(distinct variant) > 1)
having count(*) > 0

union all
select 'arm_missing', concat(source, ' test ', test_id), 1,
       'only one arm present — lift cannot be computed'
from enrollment
where variant is not null
group by source, test_id
having count(distinct variant) < 2

In [ ]:
%sql
-- ===========================================================================
-- 5. Take-rate. 'Pooled' is the all-market rollup.
-- ===========================================================================
select
  case when grouping(c.market) = 1 then 'Pooled' else c.market end as market,
  c.variant,
  count(distinct c.user_id)                                        as cohort_users,
  count(distinct s.user_id)                                        as subscribers,
  count(distinct case when s.subscription_started_with_trial
                      then s.user_id end)                          as trial_subs,
  round(100.0 * count(distinct s.user_id)
        / nullif(count(distinct c.user_id), 0), 2)                 as take_rate_pct
from cohort c
left join cohort_subs s
  on s.user_id = c.user_id and s.variant = c.variant and s.market = c.market
group by grouping sets ((c.market, c.variant), (c.variant))
order by market, c.variant

In [ ]:
%sql
-- ===========================================================================
-- 6. P&L and campaign spend.
--    net_income already nets Bolt campaign cost — do not subtract it again.
-- ===========================================================================
select
  case when grouping(c.market) = 1 then 'Pooled' else c.market end as market,
  c.variant,
  count(distinct c.user_id)                                        as cohort_users,
  count(distinct o.user_id)                                        as activated_users,
  round(100.0 * count(distinct o.user_id)
        / nullif(count(distinct c.user_id), 0), 2)                 as activation_pct,
  coalesce(sum(o.orders), 0)                                       as orders,
  round(coalesce(sum(o.gmv), 0), 2)                                as gmv,
  round(coalesce(sum(o.nmv), 0), 2)                                as nmv,
  round(coalesce(sum(o.net_income), 0), 2)                         as net_income,
  round(coalesce(sum(o.discount), 0), 2)                           as discount,
  round(coalesce(sum(o.provider_commission), 0), 2)                as provider_commission,
  round(coalesce(sum(o.bolt_delivery_cost + o.bolt_menu_cost), 0), 2)         as bolt_campaign_cost,
  round(coalesce(sum(o.provider_delivery_cost + o.provider_menu_cost), 0), 2) as provider_campaign_cost,
  round(coalesce(sum(o.bolt_delivery_cost), 0), 2)                 as bolt_delivery_cost,
  round(coalesce(sum(o.bolt_menu_cost), 0), 2)                     as bolt_menu_cost,
  round(coalesce(sum(o.provider_delivery_cost), 0), 2)             as provider_delivery_cost,
  round(coalesce(sum(o.provider_menu_cost), 0), 2)                 as provider_menu_cost,
  round(coalesce(sum(o.delivery_discount), 0), 2)                  as delivery_discount,
  round(coalesce(sum(o.menu_discount), 0), 2)                      as menu_discount,
  round(coalesce(sum(o.gmv), 0)        / nullif(count(distinct c.user_id), 0), 4) as gmv_per_user,
  round(coalesce(sum(o.orders), 0)     / nullif(count(distinct c.user_id), 0), 4) as orders_per_user,
  round(coalesce(sum(o.net_income), 0) / nullif(count(distinct c.user_id), 0), 4) as net_income_per_user,
  round(coalesce(sum(o.bolt_delivery_cost + o.bolt_menu_cost), 0)
        / nullif(count(distinct c.user_id), 0), 4)                 as bolt_cost_per_user
from cohort c
left join cohort_orders o
  on o.user_id = c.user_id and o.variant = c.variant and o.market = c.market
group by grouping sets ((c.market, c.variant), (c.variant))
order by market, c.variant

In [ ]:
%sql
-- ===========================================================================
-- 7. Treatment vs control lift, per user.
--    Per-user figures are split-invariant — draw conclusions from these rather
--    than from the absolute totals above.
-- ===========================================================================
with per_arm as (
  select
    case when grouping(c.market) = 1 then 'Pooled' else c.market end          as market,
    c.variant,
    count(distinct c.user_id)                                                 as users,
    coalesce(sum(o.gmv), 0)        / nullif(count(distinct c.user_id), 0)     as gmv_pu,
    coalesce(sum(o.orders), 0)     / nullif(count(distinct c.user_id), 0)     as orders_pu,
    coalesce(sum(o.net_income), 0) / nullif(count(distinct c.user_id), 0)     as net_pu,
    coalesce(sum(o.bolt_delivery_cost + o.bolt_menu_cost), 0)
      / nullif(count(distinct c.user_id), 0)                                  as bolt_pu,
    100.0 * count(distinct o.user_id) / nullif(count(distinct c.user_id), 0)  as activation_pct
  from cohort c
  left join cohort_orders o
    on o.user_id = c.user_id and o.variant = c.variant and o.market = c.market
  group by grouping sets ((c.market, c.variant), (c.variant))
)
select
  t.market,
  t.users                                                      as treatment_users,
  k.users                                                      as control_users,
  round(t.gmv_pu, 4)                                           as gmv_per_user_t,
  round(k.gmv_pu, 4)                                           as gmv_per_user_c,
  round(100.0 * (t.gmv_pu / nullif(k.gmv_pu, 0) - 1), 1)       as gmv_lift_pct,
  round(t.orders_pu, 4)                                        as orders_per_user_t,
  round(k.orders_pu, 4)                                        as orders_per_user_c,
  round(100.0 * (t.orders_pu / nullif(k.orders_pu, 0) - 1), 1) as orders_lift_pct,
  round(t.net_pu, 4)                                           as net_income_per_user_t,
  round(k.net_pu, 4)                                           as net_income_per_user_c,
  round(100.0 * (t.net_pu / nullif(k.net_pu, 0) - 1), 1)       as net_income_lift_pct,
  round(t.activation_pct, 2)                                   as activation_pct_t,
  round(k.activation_pct, 2)                                   as activation_pct_c,
  round(t.activation_pct - k.activation_pct, 2)                as activation_delta_pp,
  round((t.gmv_pu - k.gmv_pu) * t.users, 2)                    as incr_gmv_treated,
  round((t.orders_pu - k.orders_pu) * t.users, 2)              as incr_orders_treated,
  round((t.net_pu - k.net_pu) * t.users, 2)                    as incr_net_income_treated,
  round((t.bolt_pu - k.bolt_pu) * t.users, 2)                  as incr_bolt_cost_treated,
  case when t.bolt_pu - k.bolt_pu > 0
       then round((t.gmv_pu - k.gmv_pu) / (t.bolt_pu - k.bolt_pu), 2)
  end                                                          as uc_ratio_gmv_on_bolt_spend
from per_arm t
join per_arm k on k.market = t.market and k.variant = 'control'
where t.variant = 'treatment'
order by t.market

In [ ]:
%sql
-- ===========================================================================
-- 8. Subscription survival: share of subscribers still open today
-- ===========================================================================
select
  market,
  variant,
  count(distinct user_id)                                    as subscribers,
  count(distinct case when subscription_end_time is null
                        or subscription_end_time > current_timestamp()
                      then user_id end)                      as still_active,
  round(100.0 * count(distinct case when subscription_end_time is null
                                      or subscription_end_time > current_timestamp()
                                    then user_id end)
        / nullif(count(distinct user_id), 0), 2)             as active_pct
from cohort_subs
group by market, variant
order by market, variant

In [ ]:
%sql
-- ===========================================================================
-- 9. Behavioural retention: distinct weekly orderers.
--    Week 0 = each user's own test start, not a fixed calendar date.
-- ===========================================================================
select
  market,
  variant,
  cast(floor(datediff(order_created_date, test_start_date) / 7) as int) as week_no,
  count(distinct user_id)                                              as active_users
from cohort_order_days
group by market, variant,
         cast(floor(datediff(order_created_date, test_start_date) / 7) as int)
order by market, variant, week_no

In [ ]:
%sql
-- ===========================================================================
-- 10. Who subscribed: lifecycle stage the day before test start, and order
--     frequency in the 90 days before test start.
-- ===========================================================================
with subs as (
  select distinct user_id, market, variant, country_code, to_date(test_start) as test_start_date
  from cohort_subs
)
select
  s.market,
  s.variant,
  coalesce(l.prior_stage, 'Unknown')       as prior_stage,
  coalesce(l.prior_cohort_name, 'Unknown') as prior_cohort_name,
  case when coalesce(f.prior_orders, 0) = 0 then '0'
       when f.prior_orders between 1 and 2  then '1-2'
       when f.prior_orders between 3 and 5  then '3-5'
       else '6+' end                       as prior_freq_bucket,
  count(distinct s.user_id)                as subscribers
from subs s
left join cohort_lifecycle l
  on l.user_id = s.user_id
 and l.country_code = s.country_code
 and l.metric_date = date_sub(s.test_start_date, 1)
left join cohort_prior_orders f
  on f.user_id = s.user_id and f.variant = s.variant and f.market = s.market
group by s.market, s.variant,
         coalesce(l.prior_stage, 'Unknown'),
         coalesce(l.prior_cohort_name, 'Unknown'),
         case when coalesce(f.prior_orders, 0) = 0 then '0'
              when f.prior_orders between 1 and 2  then '1-2'
              when f.prior_orders between 3 and 5  then '3-5'
              else '6+' end
order by s.market, s.variant, subscribers desc

In [ ]:
%sql
-- ===========================================================================
-- 11. Subscribers by billing period and trial status
-- ===========================================================================
select
  market,
  variant,
  subscription_period_type,
  count(distinct user_id)                            as subscribers,
  count(distinct case when subscription_started_with_trial
                      then user_id end)              as trial_subs,
  round(avg(subscription_price_discounted_local), 4) as avg_price_local,
  round(max(subscription_price_discounted_local), 4) as max_price_local
from cohort_subs
group by market, variant, subscription_period_type
order by market, variant, subscribers desc

In [ ]:
%sql
-- ===========================================================================
-- 12. Standard (non-trial) Plus price in the same markets, for sizing the
--     post-trial revenue assumption. Window ends the day before test start.
-- ===========================================================================
select
  coalesce(m.market, upper(s.subscription_start_country_code))     as market,
  round(percentile(s.subscription_price_discounted_local, 0.5), 2) as median_price_local,
  count(*)                                                         as n
from main.core_models.fact_user_subscriptions s
left join markets m on m.country_code = lower(s.subscription_start_country_code)
where lower(s.subscription_start_country_code) in (select distinct country_code from cohort)
  and s.subscription_started_with_trial = false
  and s.subscription_period_type = 'month'
  and s.subscription_price_discounted_local > 0
  and s.subscription_start_date_utc
      between add_months((select win_start from bounds), -8)
          and date_sub((select win_start from bounds), 1)
group by coalesce(m.market, upper(s.subscription_start_country_code))
order by market

In [ ]:
%sql
-- ===========================================================================
-- 13. Historical subscriber survival benchmark. Cohorts old enough to have 12
--     observable months, to sanity-check the expected-active-months assumption.
-- ===========================================================================
select
  coalesce(m.market, upper(s.subscription_start_country_code)) as market,
  count(*)                                                     as subs,
  round(avg(datediff(coalesce(s.subscription_end_time, current_timestamp()),
                     s.subscription_start_time) / 30.4), 2)    as avg_months_active,
  round(100.0 * avg(case when s.subscription_end_time is null
        or datediff(s.subscription_end_time, s.subscription_start_time) >= 30
        then 1 else 0 end), 1)                                 as pct_ge_1mo,
  round(100.0 * avg(case when s.subscription_end_time is null
        or datediff(s.subscription_end_time, s.subscription_start_time) >= 90
        then 1 else 0 end), 1)                                 as pct_ge_3mo,
  round(100.0 * avg(case when s.subscription_end_time is null
        or datediff(s.subscription_end_time, s.subscription_start_time) >= 180
        then 1 else 0 end), 1)                                 as pct_ge_6mo,
  round(100.0 * avg(case when s.subscription_end_time is null
        or datediff(s.subscription_end_time, s.subscription_start_time) >= 365
        then 1 else 0 end), 1)                                 as pct_ge_12mo
from main.core_models.fact_user_subscriptions s
left join markets m on m.country_code = lower(s.subscription_start_country_code)
where lower(s.subscription_start_country_code) in (select distinct country_code from cohort)
  and s.subscription_start_date_utc
      between add_months((select win_start from bounds), -27)
          and add_months((select win_start from bounds), -15)
group by coalesce(m.market, upper(s.subscription_start_country_code))
order by market

In [ ]:
# ===========================================================================
# 14. Headline summary, printed as plain text.
# ===========================================================================
_rows = spark.sql("""
with per_arm as (
  select
    case when grouping(c.market) = 1 then 'Pooled' else c.market end         as market,
    c.variant,
    count(distinct c.user_id)                                                as users,
    count(distinct s.user_id)                                                as subs,
    coalesce(sum(o.gmv), 0)    / nullif(count(distinct c.user_id), 0)        as gmv_pu,
    coalesce(sum(o.orders), 0) / nullif(count(distinct c.user_id), 0)        as ord_pu
  from cohort c
  left join cohort_orders o
    on o.user_id = c.user_id and o.variant = c.variant and o.market = c.market
  left join cohort_subs s
    on s.user_id = c.user_id and s.variant = c.variant and s.market = c.market
  group by grouping sets ((c.market, c.variant), (c.variant))
)
select
  t.market,
  t.users as t_users, k.users as c_users,
  100.0 * t.subs / nullif(t.users, 0) as take_t,
  100.0 * k.subs / nullif(k.users, 0) as take_c,
  t.gmv_pu, k.gmv_pu as c_gmv_pu,
  t.ord_pu, k.ord_pu as c_ord_pu
from per_arm t
join per_arm k on k.market = t.market and k.variant = 'control'
where t.variant = 'treatment'
order by t.market
""").collect()

print(f"Tests {TEST_IDS}   window {WIN_START} → {WIN_END}\n")
print(f"{'Market':<12}{'T users':>10}{'C users':>10}{'Take T':>9}{'Take C':>9}"
      f"{'GMV/u T':>10}{'GMV/u C':>10}{'GMV lift':>10}{'Ord lift':>10}")
print("-" * 90)
for r in _rows:
    gmv_lift = (r["gmv_pu"] / r["c_gmv_pu"] - 1) * 100 if r["c_gmv_pu"] else None
    ord_lift = (r["ord_pu"] / r["c_ord_pu"] - 1) * 100 if r["c_ord_pu"] else None
    print(f"{r['market']:<12}{r['t_users']:>10,}{r['c_users']:>10,}"
          f"{r['take_t']:>8.2f}%{r['take_c']:>8.2f}%"
          f"{r['gmv_pu']:>10.2f}{r['c_gmv_pu']:>10.2f}"
          f"{(f'{gmv_lift:+.1f}%' if gmv_lift is not None else '—'):>10}"
          f"{(f'{ord_lift:+.1f}%' if ord_lift is not None else '—'):>10}")